In [ ]:
!pip install sentencepiece --quiet

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from allennlp.modules.scalar_mix import ScalarMix
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import shuffle
from tqdm import tqdm
import transformers

## Load data

In [ ]:
device = torch.device('cuda')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/text-difficulty-classification1'
except Exception:
    BASE_DIR = '.'

DATA_DIR    = os.path.join(BASE_DIR, 'data')
RESULTS_DIR = os.path.join(BASE_DIR, 'outputs', 'hierarchical_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))


def build_full_text(row):
    if 'full_text' in row and isinstance(row['full_text'], str) and len(row['full_text']) > 0:
        return row['full_text']
    return (
        f"Question: {row.get('question', '') or ''}\n"
        f"Choices: {row.get('choices', '') or ''}\n"
        f"Explanation: {row.get('lecture', '') or ''}\n"
        f"Solution: {row.get('solution', '') or ''}"
    )


X_train_all = df_train.apply(build_full_text, axis=1).values
X_test      = df_test.apply(build_full_text, axis=1).values

LABEL_MAP = {'elementary': 0, 'middle': 1, 'high': 2}
y_train_3class = df_train['education_level'].map(LABEL_MAP).values
y_test_3class  = df_test['education_level'].map(LABEL_MAP).values

print(f'Train: {X_train_all.shape} | Test: {X_test.shape}')
print('Train class dist:', dict(zip(*np.unique(y_train_3class, return_counts=True))))
print('Test  class dist:', dict(zip(*np.unique(y_test_3class,  return_counts=True))))

In [ ]:
np.random.seed(42)
samples_per_class_s1 = 1516
samples_per_class_s2 = 1516

# ── Stage 1: elementary vs. not-elementary ─────────────────────────
y_train_s1 = (y_train_3class != 0).astype(np.int64)  # 0=elem, 1=not-elem
y_test_s1  = (y_test_3class != 0).astype(np.int64)

# Balance: ~1516 elementary vs. ~1516 not-elementary (sampled from middle+high pool)
elem_idx     = np.where(y_train_3class == 0)[0]
not_elem_idx = np.where(y_train_3class != 0)[0]

n_elem     = min(len(elem_idx), samples_per_class_s1)
n_not_elem = min(len(not_elem_idx), samples_per_class_s1)

s1_idx = np.concatenate([
    np.random.choice(elem_idx, n_elem, replace=False),
    np.random.choice(not_elem_idx, n_not_elem, replace=False),
])
np.random.shuffle(s1_idx)

X_train_s1 = X_train_all[s1_idx]
y_train_s1_bal = y_train_s1[s1_idx]

print('Stage 1 (elem vs. not-elem):')
print(f'  Train: {X_train_s1.shape} | Class dist:',
      dict(zip(*np.unique(y_train_s1_bal, return_counts=True))))
print(f'  Test : {X_test.shape}     | Class dist:',
      dict(zip(*np.unique(y_test_s1, return_counts=True))))

# ── Stage 2: middle vs. high (subset of train + test) ──────────────
mid_idx  = np.where(y_train_3class == 1)[0]
high_idx = np.where(y_train_3class == 2)[0]

n_mid  = min(len(mid_idx), samples_per_class_s2)
n_high = min(len(high_idx), samples_per_class_s2)

s2_idx = np.concatenate([
    np.random.choice(mid_idx, n_mid, replace=False),
    np.random.choice(high_idx, n_high, replace=False),
])
np.random.shuffle(s2_idx)

X_train_s2 = X_train_all[s2_idx]
y_train_s2_bal = (y_train_3class[s2_idx] == 2).astype(np.int64)  # 0=mid, 1=high

# For Stage 2 evaluation, we want metrics on the test middle+high samples
test_mid_high_mask = (y_test_3class != 0)
X_test_s2 = X_test[test_mid_high_mask]
y_test_s2 = (y_test_3class[test_mid_high_mask] == 2).astype(np.int64)

print('\nStage 2 (middle vs. high):')
print(f'  Train: {X_train_s2.shape} | Class dist:',
      dict(zip(*np.unique(y_train_s2_bal, return_counts=True))))
print(f'  Test : {X_test_s2.shape}  | Class dist:',
      dict(zip(*np.unique(y_test_s2, return_counts=True))))

## Model definition

Same `ScoringModel` as before, parameterized for binary or 3-class output. Reused for both stages.

In [ ]:
class ScoringModel(torch.nn.Module):
    def __init__(self, language_model, prefix, num_classes=2) -> None:
        super().__init__()
        self.prefix = prefix

        self.tokenizer = AutoTokenizer.from_pretrained(language_model)
        self.lm = AutoModel.from_pretrained(language_model).to(device)
        self.scalar_mix = ScalarMix(self.lm.config.num_hidden_layers + 1)
        self.dropout = torch.nn.Dropout(p=0.2)
        self.lm_name = language_model

        self.classification_head = torch.nn.Sequential(
            torch.nn.Linear(self.lm.config.hidden_size, self.lm.config.hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(self.lm.config.hidden_size, num_classes),
        )

        self.loss = torch.nn.CrossEntropyLoss()

        self.X = None;  self.y = None
        self.eval_X = None; self.eval_y = None

        # If set, every epoch summary / eval report is appended to this file
        # in addition to being printed to stdout.
        self.log_path = None

    def set_dataset(self, X, y):
        self.X = X; self.y = y

    def set_evalset(self, X, y):
        self.eval_X = X; self.eval_y = y

    def _log(self, msg=""):
        """Print to stdout and append to self.log_path (if set)."""
        print(msg)
        if self.log_path is not None:
            with open(self.log_path, "a") as f:
                f.write(str(msg) + "\n")

    def forward(self, input_text):
        inputs = self.tokenizer(
            input_text, return_tensors='pt', padding=True, truncation=True,
            max_length=self.lm.config.max_position_embeddings - 2,
        )
        outputs = self.lm(**inputs.to(device), output_hidden_states=True)
        pooled = torch.mean(
            self.dropout(self.scalar_mix(outputs.hidden_states)),
            dim=1,
        )
        return self.classification_head(pooled)

    def predict_all(self, texts):
        """Return predicted classes for a list of texts (one at a time, low memory)."""
        self.eval()
        preds = []
        with torch.no_grad():
            for t in tqdm(texts, desc='predict'):
                logits = self.forward(t)
                preds.append(logits.argmax(dim=1).item())
        return np.array(preds)

    def self_eval(self, target_names=None):
        if target_names is None:
            target_names = [f'class_{i}' for i in range(2)]
        self.eval()
        predictions = []
        with torch.no_grad():
            for text in tqdm(self.eval_X, desc='eval'):
                logits = self.forward(text)
                predictions.append(logits.argmax(dim=1).item())

        true_labels = self.eval_y
        if torch.is_tensor(true_labels):
            true_labels = true_labels.cpu().numpy()

        report = classification_report(true_labels, predictions, target_names=target_names)
        self._log(report)
        return {'macro_f1': f1_score(true_labels, predictions, average='macro')}

    def fit(self, epochs, optimizer, scheduler, batch_size=4):
        self.train()
        skipped_total = 0
        for epoch in range(epochs):
            self._log(f'\nEpoch {epoch}')
            r, num_s, skipped = 0.0, 0.0, 0
            d, d_y = shuffle(self.X, self.y)

            batches_X = [d[n:n+batch_size] for n in range(0, len(d), batch_size)]
            batches_y = [d_y[n:n+batch_size] for n in range(0, len(d_y), batch_size)]

            for b in tqdm(range(len(batches_X))):
                pred = self.forward(list(batches_X[b]))
                ls = self.loss(pred, batches_y[b])

                if not torch.isfinite(ls):
                    optimizer.zero_grad()
                    skipped += 1
                    continue

                optimizer.zero_grad()
                ls.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

                r += ls.detach().item()
                num_s += 1

                if b % 50 == 0 and b > 0:
                    print(f'  loss: {r / num_s:.4f}')

            self._log(f'  Epoch {epoch}: {skipped} skipped, avg loss {r / max(num_s, 1):.4f}')
            skipped_total += skipped

            if self.eval_X is not None:
                ev = self.self_eval()
                self._log(ev)
                self.train()

        self._log(f'\nTotal skipped: {skipped_total}')


## Helper: build a fresh DeBERTa model with the standard stability setup

In [ ]:
LM_NAME = 'microsoft/deberta-v3-large'
EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-6


def build_model(num_classes, prefix):
    model = ScoringModel(LM_NAME, prefix=prefix, num_classes=num_classes).to(device)
    model.lm = model.lm.to(torch.float32)
    model.lm.gradient_checkpointing_enable()
    model.lm.config.use_cache = False
    return model


def make_optimizer_and_scheduler(model, n_train_samples):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, eps=1e-6)
    total_steps = EPOCHS * (n_train_samples // BATCH_SIZE + 1)
    scheduler = transformers.get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    return optimizer, scheduler

## Train Stage 1: elementary vs. not-elementary

In [ ]:
print('=' * 70)
print('  STAGE 1: elementary vs. not-elementary')
print('=' * 70)

torch.manual_seed(42)
np.random.seed(42)

y_train_s1_tensor = torch.tensor(y_train_s1_bal, dtype=torch.long).to(device)
y_test_s1_tensor  = torch.tensor(y_test_s1,      dtype=torch.long).to(device)

model_s1 = build_model(num_classes=2, prefix='stage1')
model_s1.set_dataset(X_train_s1, y_train_s1_tensor)
model_s1.set_evalset(X_test, y_test_s1_tensor)

# Log everything fit() / self_eval() prints to a per-stage file as well as stdout.
stage1_log_path = os.path.join(RESULTS_DIR, 'stage1_training_log.txt')
# Truncate any prior log so this run starts fresh.
open(stage1_log_path, 'w').close()
model_s1.log_path = stage1_log_path

optimizer_s1, scheduler_s1 = make_optimizer_and_scheduler(model_s1, len(X_train_s1))
print(f'Total steps: {EPOCHS * (len(X_train_s1) // BATCH_SIZE + 1)}')

model_s1.fit(EPOCHS, optimizer_s1, scheduler_s1, batch_size=BATCH_SIZE)

model_s1._log('\n--- Final Stage 1 evaluation ---')
s1_final = model_s1.self_eval(target_names=['elementary', 'not_elementary'])
model_s1._log(s1_final)

torch.save(model_s1.state_dict(), os.path.join(RESULTS_DIR, 'stage1_elem_vs_not.pt'))
print(f'\n[Stage 1 log written to: {stage1_log_path}]')


## Train Stage 2: middle vs. high

In [ ]:
# Free Stage 1 model from GPU before loading Stage 2
del model_s1, optimizer_s1, scheduler_s1
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('=' * 70)
print('  STAGE 2: middle vs. high')
print('=' * 70)

torch.manual_seed(42)
np.random.seed(42)

y_train_s2_tensor = torch.tensor(y_train_s2_bal, dtype=torch.long).to(device)
y_test_s2_tensor  = torch.tensor(y_test_s2,      dtype=torch.long).to(device)

model_s2 = build_model(num_classes=2, prefix='stage2')
model_s2.set_dataset(X_train_s2, y_train_s2_tensor)
model_s2.set_evalset(X_test_s2, y_test_s2_tensor)

# Log everything fit() / self_eval() prints to a per-stage file as well as stdout.
stage2_log_path = os.path.join(RESULTS_DIR, 'stage2_training_log.txt')
open(stage2_log_path, 'w').close()
model_s2.log_path = stage2_log_path

optimizer_s2, scheduler_s2 = make_optimizer_and_scheduler(model_s2, len(X_train_s2))

model_s2.fit(EPOCHS, optimizer_s2, scheduler_s2, batch_size=BATCH_SIZE)

model_s2._log('\n--- Final Stage 2 evaluation ---')
s2_final = model_s2.self_eval(target_names=['middle', 'high'])
model_s2._log(s2_final)

torch.save(model_s2.state_dict(), os.path.join(RESULTS_DIR, 'stage2_mid_vs_high.pt'))
print(f'\n[Stage 2 log written to: {stage2_log_path}]')


## Combine: hierarchical inference on the full test set

For each test sample:
1. Run Stage 1. If it predicts 0 (elementary), final prediction = 0.
2. Otherwise, run Stage 2. If Stage 2 predicts 0 (middle), final = 1; if 1 (high), final = 2.

In [ ]:
# Reload Stage 1 weights (we deleted the model to free memory)
model_s1 = build_model(num_classes=2, prefix='stage1_eval')
model_s1.load_state_dict(torch.load(os.path.join(RESULTS_DIR, 'stage1_elem_vs_not.pt')))
model_s1.eval()

# Stage 1 predictions on every test sample
print('Stage 1 inference on full test set...')
s1_preds = model_s1.predict_all(list(X_test))

# Free Stage 1 again before loading Stage 2
del model_s1
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_s2 = build_model(num_classes=2, prefix='stage2_eval')
model_s2.load_state_dict(torch.load(os.path.join(RESULTS_DIR, 'stage2_mid_vs_high.pt')))
model_s2.eval()

# Stage 2 predictions only on samples Stage 1 routed to "not-elementary"
not_elem_mask = (s1_preds == 1)
not_elem_texts = X_test[not_elem_mask]

print(f'\nStage 2 inference on {not_elem_mask.sum()} routed samples...')
s2_preds_subset = model_s2.predict_all(list(not_elem_texts))

# Combine into final 3-class predictions
final_preds = np.zeros(len(X_test), dtype=np.int64)
# Elementary samples (Stage 1 said 0) stay at 0
# Non-elementary samples get 1 (middle) if Stage 2 said 0, else 2 (high)
final_preds[not_elem_mask] = s2_preds_subset + 1

# ── Logging helper: print AND append to the hierarchical results file ────
hierarchical_log_path = os.path.join(RESULTS_DIR, 'hierarchical_final_results.txt')
open(hierarchical_log_path, 'w').close()  # truncate any prior file

def hlog(msg=""):
    print(msg)
    with open(hierarchical_log_path, 'a') as f:
        f.write(str(msg) + '\n')

# Final report on 3-class problem
hlog('\n=== HIERARCHICAL CLASSIFIER FINAL RESULTS ===')
hlog(classification_report(
    y_test_3class, final_preds,
    target_names=['elementary', 'middle', 'high'],
))
final_macro_f1 = f1_score(y_test_3class, final_preds, average='macro')
hlog(f'\nFinal macro-F1: {final_macro_f1:.4f}')

# Confusion matrix to see where errors are
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_3class, final_preds)
hlog('\nConfusion matrix (rows=true, cols=predicted):')
hlog('               elem  mid  high')
hlog(f'  elementary  {cm[0][0]:4d}  {cm[0][1]:4d}  {cm[0][2]:4d}')
hlog(f'  middle      {cm[1][0]:4d}  {cm[1][1]:4d}  {cm[1][2]:4d}')
hlog(f'  high        {cm[2][0]:4d}  {cm[2][1]:4d}  {cm[2][2]:4d}')

print(f'\n[Hierarchical results written to: {hierarchical_log_path}]')